# Fine-tune PointPillars on Livox Mid-360S data (Colab, Python-3.10 env)

**Overfit pipeline-proof run:** train on all 48 hand-labeled frames (no val, no eval) and watch the loss fall toward ~0 — proving the data->label->model pipeline end-to-end before real tuning.

### ⚠️ Read this before running
This notebook uses a **Python-3.10 conda environment** so spconv installs from a **prebuilt wheel in seconds** (Colab's default Python 3.12 has no spconv wheel and would compile it from source for ~3 hours).

- **Cell 1 installs condacolab and RESTARTS the kernel.** Run **Cell 1 alone first**; you will see a **"⚠️ Your session crashed"** message — **that is normal** (it's the restart). Then continue from Cell 2 downward.
- After the restart, run the rest **top-to-bottom** ("Run all" also works once Cell 1 has restarted).

### Prerequisites
- **GPU runtime:** Runtime -> Change runtime type -> Hardware accelerator -> **GPU**.
- **Data bundle in Drive root:** `/content/drive/MyDrive/perception_colab_data.zip` (48 `.bin` + labels + `pointpillar_7728.pth`).

### Pinned environment (do not change — see AUDIT_colab_env)
Python **3.10** · torch **2.1.2+cu118** · numpy **1.23.5** (`<1.24`, mandatory) · **spconv-cu118** (prebuilt cp310) · OpenPCDet **846cf3e** + `patch_openpcdet.py`.

## 1. Install condacolab (RUN THIS CELL ALONE — the kernel restarts)

⚠️ Run **only this cell** first. The kernel will restart and you'll see **"Your session crashed for an unknown reason"** — **that is expected** (it's condacolab switching the runtime to Python 3.10). After it restarts, continue from Cell 2. Do **not** put anything else in this cell.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()   # kernel RESTARTS here — "session crashed" is EXPECTED

## 2. Environment sanity + pinned installs + the decisive no-compile check

Installs the pinned stack and **verifies spconv imports in seconds** (prebuilt wheel). If spconv instead starts compiling (ninja / hundreds of files), **stop** — the prebuilt wheel wasn't used (wrong Python/wheel).

In [ ]:
import condacolab; condacolab.check()
!python --version                      # expect Python 3.10.x

!pip install -q torch==2.1.2+cu118 torchvision==0.16.2+cu118 --index-url https://download.pytorch.org/whl/cu118
!pip install -q "numpy==1.23.5"
!pip install -q spconv-cu118

import time; t = time.time()
import spconv.pytorch as spconv
dt = time.time() - t
print(f"spconv {spconv.__version__} imported in {dt:.1f}s")
assert dt < 60, "spconv took too long — prebuilt wheel NOT used (wrong python/wheel). STOP."

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > GPU, re-run from Cell 1."

## 3. Clone our repo + mount Drive + unzip data

In [ ]:
!git clone https://github.com/Abdulrahman2200925/perception.git /content/perception
from google.colab import drive; drive.mount('/content/drive')
import os
ZIP = '/content/drive/MyDrive/perception_colab_data.zip'
assert os.path.exists(ZIP), f'zip not found at {ZIP} — upload it to Drive root.'
!unzip -q -o "$ZIP" -d /content/perception
import glob, json
bins = glob.glob('/content/perception/data/derived/bin/**/*.bin', recursive=True)
man  = '/content/perception/data/derived/labels_openpcdet/MANIFEST_pairing.json'
ckpt = '/content/perception/checkpoints/pretrained/pointpillar_7728.pth'
print('bin files:', len(bins))
assert len(bins) == 48, f'expected 48 bins, found {len(bins)}'
assert os.path.exists(man) and os.path.exists(ckpt)
print('manifest frames:', len(json.load(open(man))['frames']), '| data OK')

## 4. Clone + PATCH + build OpenPCDet (846cf3e)

`patch_openpcdet.py` fixes THC / `AT_CHECK` / `.data<T>()` for torch 2.x — **compiled ops only; the PointPillar ONNX graph is unaffected**. `requirements.txt` may bump numpy >= 1.24, which would crash training at `np.int` (`base_bev_backbone.py:60`), so numpy is **re-pinned to 1.23.5 after** requirements (Cell 5 verifies).

In [ ]:
!git clone https://github.com/open-mmlab/OpenPCDet.git /content/OpenPCDet
%cd /content/OpenPCDet
!git checkout 846cf3e
!python /content/perception/colab_bundle/patch_openpcdet.py /content/OpenPCDet
!pip install -q -r requirements.txt
!pip install -q "numpy==1.23.5"        # RE-PIN: requirements may bump numpy>=1.24 (crashes at np.int)
!python setup.py develop

## 5. Confirm build + versions (guard rail)

In [ ]:
import pcdet; print("pcdet", pcdet.__version__)
import spconv.pytorch; print("spconv.pytorch OK")
import numpy; print("numpy", numpy.__version__)
assert numpy.__version__.startswith("1.23"), "numpy got bumped — re-pin 1.23.5 and re-run Cell 4/5"

## 6. Inject our dataset class + configs into OpenPCDet

In [ ]:
import os, shutil, re

# 5a. dataset class -> pcdet/datasets/mid360/
dst_dir = '/content/OpenPCDet/pcdet/datasets/mid360'
os.makedirs(dst_dir, exist_ok=True)
open(os.path.join(dst_dir, '__init__.py'), 'a').close()
shutil.copy('/content/perception/mine/dataset/mid360_dataset.py',
            os.path.join(dst_dir, 'mid360_dataset.py'))

# 5b. register Mid360Dataset in pcdet/datasets/__init__.py (idempotent)
init_p = '/content/OpenPCDet/pcdet/datasets/__init__.py'
src = open(init_p).read()
if 'Mid360Dataset' not in src:
    src = src.replace('from .dataset import DatasetTemplate',
                      'from .dataset import DatasetTemplate\nfrom .mid360.mid360_dataset import Mid360Dataset', 1)
    src = src.replace('__all__ = {',
                      "__all__ = {\n    'Mid360Dataset': Mid360Dataset,", 1)
    open(init_p, 'w').write(src)
    print('registered Mid360Dataset')
else:
    print('Mid360Dataset already registered')
assert 'Mid360Dataset' in open(init_p).read()

# 5c. copy configs into OpenPCDet tools/cfgs + fix Colab paths
os.makedirs('/content/OpenPCDet/tools/cfgs/mid360_models', exist_ok=True)
ds_dst = '/content/OpenPCDet/tools/cfgs/dataset_configs/mid360_dataset.yaml'
md_dst = '/content/OpenPCDet/tools/cfgs/mid360_models/mid360_pointpillar.yaml'
shutil.copy('/content/perception/configs/mid360_dataset.yaml', ds_dst)
shutil.copy('/content/perception/configs/mid360_pointpillar.yaml', md_dst)

# dataset cfg: DATA_PATH -> /content/perception
s = open(ds_dst).read()
s = re.sub(r"DATA_PATH:.*", "DATA_PATH: '/content/perception'", s, count=1)
open(ds_dst, 'w').write(s)

# model cfg: _BASE_CONFIG_ -> the copied dataset cfg (relative to tools/)
s = open(md_dst).read()
s = re.sub(r"_BASE_CONFIG_:.*", "_BASE_CONFIG_: cfgs/dataset_configs/mid360_dataset.yaml", s, count=1)
open(md_dst, 'w').write(s)

print('DATA_PATH    ->', [l for l in open(ds_dst) if l.strip().startswith('DATA_PATH')][0].strip())
print('_BASE_CONFIG_->', [l for l in open(md_dst) if '_BASE_CONFIG_' in l][0].strip())

## 7. Full smoke test (voxelization + collate)

In [ ]:
from pathlib import Path
import yaml
from easydict import EasyDict
from pcdet.utils import common_utils
from pcdet.datasets.mid360.mid360_dataset import Mid360Dataset

dcfg = EasyDict(yaml.safe_load(open('/content/OpenPCDet/tools/cfgs/dataset_configs/mid360_dataset.yaml')))
logger = common_utils.create_logger()
try:
    ds = Mid360Dataset(dataset_cfg=dcfg, class_names=['Car','Pedestrian','Cyclist'],
                       training=True, root_path=Path('/content/perception'), logger=logger)
    print('dataset len:', len(ds))
    d = ds[0]
    print('data_dict keys:', list(d.keys()))
    for k in ['points','voxels','voxel_coords','voxel_num_points','gt_boxes']:
        if k in d:
            print(f'  {k}: shape={getattr(d[k], "shape", None)}')
    print('frame_id:', d.get('frame_id'))
    assert 'voxels' in d, 'voxelization did not run'
    assert d['gt_boxes'].shape[1] == 8, f"gt_boxes should be (M,8), got {d['gt_boxes'].shape}"
    batch = ds.collate_batch([ds[0], ds[1]])
    print('collated batch keys:', list(batch.keys()), '| batch_size:', batch['batch_size'])
    print('SMOKE TEST PASS - full pipeline (load->augment->encode->voxelize->collate) OK')
except Exception as e:
    import traceback; traceback.print_exc()
    raise SystemExit('SMOKE TEST FAILED - fix before training. See error above.')

## 8. Overfit training (fine-tune, no eval)

Fine-tunes from `pointpillar_7728.pth` on all 48 frames. **Success = training loss falling toward ~0.** `Mid360Dataset` has no `evaluation()`; `846cf3e`'s `train.py` runs a post-training eval by default that **raises AFTER the checkpoint is saved** — expected and ignorable (the loss curve is the signal; Cell 9 copies the saved checkpoint regardless).

In [ ]:
%cd /content/OpenPCDet/tools
!python train.py --cfg_file cfgs/mid360_models/mid360_pointpillar.yaml \
    --pretrained_model /content/perception/checkpoints/pretrained/pointpillar_7728.pth \
    --batch_size 2 --epochs 80 --workers 2

## 9. Save checkpoint to Drive (portable format for the Jetson)

Copies the full checkpoint (resume/debug) **and** writes `mid360_portable.pth` — just the `model_state`, saved in the **old serialization format** so it loads in the Jetson / NVIDIA `846cf3e` + torch-1.x **export** environment (per AUDIT_colab_env STEP 5).

In [ ]:
import glob, torch, os, shutil
ck = sorted(glob.glob('/content/OpenPCDet/output/**/ckpt/*.pth', recursive=True))
assert ck, "no checkpoint found — did training run?"
latest = ck[-1]; print("latest:", latest)
os.makedirs('/content/drive/MyDrive/perception_ckpts', exist_ok=True)
# full ckpt (resume/debug)
shutil.copy(latest, '/content/drive/MyDrive/perception_ckpts/')
# PORTABLE weights for the Jetson/NVIDIA export env (old serialization -> any torch)
c = torch.load(latest, map_location='cpu')
sd = c.get('model_state', c)
torch.save(sd, '/content/drive/MyDrive/perception_ckpts/mid360_portable.pth',
           _use_new_zipfile_serialization=False, pickle_protocol=2)
print("saved portable weights -> /content/drive/MyDrive/perception_ckpts/mid360_portable.pth")

## 10. Next steps

- **Good result:** the training loss decreases steadily toward ~0 over the 80 epochs — the pipeline is proven on the target sensor.
- **Next phase (NOT in this notebook):** ONNX export in NVIDIA's validated **846cf3e + torch-1.x** environment on `mid360_portable.pth` -> sync `params.h` (range / voxel / anchor bottom-heights) -> build the TensorRT `.plan` on the Jetson.
- After the overfit proof: collect/label more data, add a real train/val split + `evaluation()`, re-tune anchors / LR.